In [0]:
RAW_DATA_PATH = "/Volumes/marathos/default/raw/TWO_CENTURIES_OF_UM_RACES.csv"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv(RAW_DATA_PATH))

display(df)

In [0]:
num_rows = df.count()
num_cols = len(df.columns)

print(f"Antal rader:    {num_rows:,}")
print(f"Antal kolumner: {num_cols}")

In [0]:
df.printSchema()

In [0]:
df.describe().display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

null_counts.display()

In [0]:
unique_events = df.select("Event name").distinct().count()
print(f"Antal unika event: {unique_events:,}")

# Åldersfördelning bland löparna

In [0]:
from pyspark.sql.functions import col

df_age = df.withColumn(
    "age",
    (col("Year of event") - col("Athlete year of birth")).cast("integer")
)

df_age_clean = df_age.filter((col("age") >= 18) & (col("age") <= 90))
df_age_clean.groupBy("age").count().orderBy("age").display()

In [0]:
from pyspark.sql.functions import desc

df.groupBy("Athlete country") \
  .count() \
  .orderBy(desc("count")) \
  .limit(20) \
  .display()